# 02. PyTorch 기초

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_02_PyTorch_%EA%B8%B0%EC%B4%88.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## 실행 환경과 GPU


In [ ]:
import torch

print(torch.__version__)
print("cuda available:", torch.cuda.is_available())

# 권장 패턴: GPU가 있으면 GPU, 없으면 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

x = torch.rand(1000, 1000).to(device)   # 텐서를 해당 장치로 이동
y = torch.rand(1000, 1000, device=device)
z = x @ y                                # 행렬곱은 GPU에서 대규모 병렬 처리
print("result device:", z.device)

In [ ]:
import torch, numpy as np, random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)   # 멀티 GPU 포함
    # 완전한 재현이 필요하면 아래도 설정(속도는 다소 느려질 수 있음)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## 텐서 생성과 속성


In [ ]:
import torch

# (1) 직접 생성
x = torch.tensor([1, 2, 3])          # dtype은 값에서 추론 -> int64
xf = torch.tensor([1.0, 2.0, 3.0])   # 소수점이 있으면 float32

# (2) 초기화 생성
zeros = torch.zeros(3, 4)            # 3x4 영행렬(모두 0)
ones  = torch.ones(2, 3)            # 모두 1
eye   = torch.eye(3)                 # 단위행렬(대각선만 1)

# (3) 난수 생성
rand  = torch.rand(2, 2)            # 균등분포 [0, 1): 0~1 사이를 고르게
randn = torch.randn(2, 2)          # 표준정규분포 N(0, 1): 0 근처가 많음

# (4) NumPy 상호운용
import numpy as np
a = np.array([[1.0, 2.0], [3.0, 4.0]])
t = torch.from_numpy(a)            # CPU에서는 메모리를 공유

# --- 세 가지 속성 확인 ---
x = torch.rand(3, 4)
print(x.shape)   # torch.Size([3, 4])
print(x.dtype)   # torch.float32
print(x.device)  # cpu 또는 cuda:0

## 인덱싱·슬라이싱·브로드캐스팅


In [ ]:
import torch
x = torch.arange(0, 12).reshape(3, 4)   # 0~11을 3행 4열로 배치
print(x[0])        # 첫 번째 행 -> tensor([0, 1, 2, 3])
print(x[:, 1])     # 두 번째 열 전체 -> tensor([1, 5, 9])
print(x[1:, 2:])   # 2행부터, 2열부터(아래쪽 2x2 부분 행렬)
print(x[x > 5])    # 불리언 마스크: 5보다 큰 값만 골라서 1차원으로

In [ ]:
import torch
x = torch.arange(0, 12).reshape(3, 4)        # (3, 4)
bias = torch.tensor([1, 10, 100, 1000])      # (4,) -> 내부적으로 (1, 4)로 해석
print(x + bias)    # (3, 4) + (4,) -> (3, 4): 각 행에 같은 bias가 더해짐

col = torch.tensor([[0], [10], [20]])        # (3, 1)
print(x + col)     # (3, 4) + (3, 1) -> (3, 4): 각 열에 같은 col이 더해짐

## 자동 미분(Autograd)

### 📝 예제 — Autograd로 도함수 값 계산

\(y=x^2+3x+1\)을 \(x=2\)에서 미분한 값을 PyTorch로 구하고, 우리가 손으로 구한 결과와 비교한다.


In [ ]:
x = torch.tensor([2.0], requires_grad=True)  # 기울기 추적을 켠다
y = x**2 + 3*x + 1                            # 순전파(그래프가 기록됨)
y.backward()                                  # 역전파(기울기 계산)
print(x.grad)                                 # tensor([7.])

### 📝 예제 — 다변수 기울기: 두 파라미터에 대한 편미분

④ 코드로도 동일하게 나온다. 기울기가 음수라는 것은 "\(w,b\)를 키우면 손실이 줄어든다"는 뜻이고, 실제로 예측 2가 정답 5보다 작으니 "더 키워야 한다"는 직관과 정확히 맞는다.


In [ ]:
w = torch.tensor([1.0], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
x, t = torch.tensor([2.0]), torch.tensor([5.0])
L = (w*x + b - t)**2
L.backward()
print(w.grad, b.grad)   # tensor([-12.]) tensor([-6.])

## 학습 루프(Training Loop)


In [ ]:
import torch
import torch.nn as nn

# y = 2x 를 학습하는 가장 단순한 예제
x = torch.tensor([[1.0], [2.0], [3.0], [4.0]])
y = torch.tensor([[2.0], [4.0], [6.0], [8.0]])

model = nn.Linear(1, 1)                  # 선형 모델 y = wx + b
criterion = nn.MSELoss()                 # 손실함수(채점 기준)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)  # 갱신 담당

for epoch in range(200):
    pred = model(x)                      # (1) 순전파: 예측 만들기
    loss = criterion(pred, y)            # (2) 손실 계산: 얼마나 틀렸나
    optimizer.zero_grad()                # 이전 grad 누적 방지
    loss.backward()                      # (3) 역전파: 기울기 구하기
    optimizer.step()                     # (4) 가중치 갱신: 한 발 이동

print("w, b:", model.weight.item(), model.bias.item())

### 🔬 실습 — 전체 학습 루프를 한 번에 읽어 보기

지금까지의 조각들을 합치면 실제 학습 코드가 된다. 아래는 "데이터를 학습용과 검증용으로 나누고, 여러 epoch 동안 학습하면서 매 epoch 검증 성능을 확인"하는 가장 표준적인 형태다. 주석을 따라가며 각 줄이 앞서 배운 어떤 개념에 해당하는지 짚어 보자.


In [ ]:
import torch
import torch.nn as nn

# (1) 장치 결정: GPU가 있으면 GPU, 없으면 CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# (2) 모델, 손실함수, optimizer 준비
model = nn.Linear(4, 3).to(device)   # 입력 4차원 -> 3개 클래스 점수
criterion = nn.CrossEntropyLoss()    # 분류용 손실
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# (3) 학습/검증 데이터 (예시용 가짜 데이터)
X_train = torch.randn(80, 4)
y_train = torch.randint(0, 3, (80,))   # 0,1,2 중 하나의 정답
X_val   = torch.randn(20, 4)
y_val   = torch.randint(0, 3, (20,))

for epoch in range(10):                # (4) 같은 데이터를 10번 반복 학습
    # ----- 학습 단계 -----
    model.train()                      # 학습 모드로 전환
    optimizer.zero_grad()              # 이전 기울기 비우기
    out = model(X_train.to(device))    # 순전파: 점수 계산
    loss = criterion(out, y_train.to(device))  # 손실 계산
    loss.backward()                    # 역전파: 기울기 계산
    optimizer.step()                   # 파라미터 갱신

    # ----- 검증 단계 -----
    model.eval()                       # 평가 모드로 전환
    with torch.no_grad():              # 기울기 추적 끔(빠르고 가벼움)
        val_out = model(X_val.to(device))
        pred = val_out.argmax(dim=1)   # 가장 점수 높은 클래스 = 예측
        acc = (pred == y_val.to(device)).float().mean()  # 정확도

    print(f"epoch {epoch+1:2d} | loss {loss.item():.3f} | val acc {acc.item():.3f}")

## 모델 저장과 불러오기


In [ ]:
# --- 저장 ---
torch.save(model.state_dict(), "linear.pt")

# --- 불러오기: 동일 구조 모델을 만든 뒤 파라미터를 주입 ---
model2 = nn.Linear(4, 3)   # 위에서 학습된 model과 동일 구조로 생성
model2.load_state_dict(torch.load("linear.pt"))
model2.eval()          # 평가 모드 전환(매우 중요)

긴 학습을 중간에 멈췄다가 *이어서* 재개하려면 모델 가중치뿐 아니라 optimizer의 상태(Adam의 모멘트 등)와 현재 epoch, 손실 이력도 함께 저장해야 한다. 가중치만 저장하면 "어디까지 학습했는지, 어떤 속도로 가고 있었는지"를 잃어버린다. 보통 다음처럼 딕셔너리로 묶어 저장한다.


In [ ]:
torch.save({
    "epoch": epoch,
    "model": model.state_dict(),
    "optim": optimizer.state_dict(),
    "loss": loss.item(),
}, "ckpt.pt")

실전에서 자주 겪는 상황이 있다. GPU 서버에서 학습해 저장한 모델을, GPU가 없는 노트북이나 추론 서버에서 불러오려 하면 "CUDA 장치를 찾을 수 없다"는 오류가 난다. 저장된 파라미터에 "원래 GPU에 있었다"는 정보가 함께 적혀 있기 때문이다. 이때는 불러올 때 `map_location`으로 "이 장치로 옮겨서 읽으라"고 지시하면 된다.


In [ ]:
# 예시를 위해 현재 model을 먼저 저장해 둔다(원래는 GPU 서버에서 저장된 파일)
torch.save(model.state_dict(), "model.pt")

# CPU만 있는 환경에서 GPU로 저장된 모델 읽기
state = torch.load("model.pt", map_location="cpu")
model.load_state_dict(state)

# 또는 현재 환경에 맞춰 자동으로
device = "cuda" if torch.cuda.is_available() else "cpu"
state = torch.load("model.pt", map_location=device)
model.load_state_dict(state)

## 실습: 군사 장비 분류 데이터 전처리

### 🔬 실습 — 장비 센서 데이터셋과 DataLoader 구성

① **Dataset 정의**: 입력 특성 \([온도,진동,압력,전류]\), 레이블 0(정상)/1(경고)/2(위험). `Dataset`을 만들려면 두 가지 약속만 지키면 된다: `__len__`(데이터가 몇 개인지)과 `__getitem__`(`idx`번째 데이터를 어떻게 돌려줄지). 실제 프로젝트에서는 `__init__`에서 CSV 파일을 읽어 텐서로 바꾸지만, 여기서는 난수로 모사한다.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class EquipmentDataset(Dataset):
    def __init__(self, n=1000):
        self.data   = torch.randn(n, 4)             # 샘플 n개, 특성 4개
        self.labels = torch.randint(0, 3, (n,))     # 클래스 0, 1, 2
    def __len__(self):
        return len(self.data)                       # 데이터 개수
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]     # (입력, 정답) 한 쌍

② **표준화(정규화)**: 특성마다 단위와 범위가 달라(온도 $$수십, 진동 $$0.0x) 그대로 학습하면 큰 값이 손실을 지배한다(목소리 큰 사람만 회의에서 들리는 격). 평균 0, 표준편차 1로 맞추면 모든 특성이 비슷한 크기가 되어 학습이 안정된다. 통계량(평균\,$$\,표준편차)은 반드시 *학습 데이터*에서 구해 검증/테스트에 동일하게 적용한다. 검증 데이터로 평균을 구하면 "아직 보지 않아야 할 답"을 미리 엿보는 셈이라 평가가 부풀려진다(정보 누출, leakage).


In [ ]:
dataset = EquipmentDataset()                    # 표준화할 데이터셋 준비
mean = dataset.data.mean(dim=0, keepdim=True)   # 특성별 평균 (1, 4)
std  = dataset.data.std(dim=0, keepdim=True) + 1e-8  # 0으로 나눔 방지
dataset.data = (dataset.data - mean) / std      # 브로드캐스팅으로 표준화

③ **DataLoader로 미니배치 구성**: 학습 시 순서를 섞어(`shuffle=True`) 일반화를 돕고, `batch_size`만큼 묶어 공급한다.


In [ ]:
dataset = EquipmentDataset()
loader  = DataLoader(dataset, batch_size=32, shuffle=True)

for batch_data, batch_labels in loader:
    print("data shape :", batch_data.shape)
    print("label shape:", batch_labels.shape)
    break   # 첫 배치만 확인하고 멈춤

### 🔬 실습 — 장비 상태 분류기 학습시키기

⑤ **학습/검증 데이터 나누기**: 가진 데이터를 학습용과 검증용으로 쪼갠다. 보통 \(8:2\)나 \(7:3\)으로 나눈다. 검증 데이터는 "학습에는 절대 쓰지 않고, 일반화 성능만 확인"하는 용도다. 시험 직전에 풀어 보는 *모의고사*와 같아서, 실제 시험(배포 환경) 전에 실력을 가늠하게 해 준다.


In [ ]:
from torch.utils.data import random_split

full = EquipmentDataset(n=1000)
n_val = 200
n_train = len(full) - n_val
train_set, val_set = random_split(full, [n_train, n_val])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=32)   # 검증은 안 섞음

⑥ **모델$$손실$$optimizer 준비**: 입력 4개(온도$$진동$$압력$$전류), 출력 3클래스(정상/경고/위험)이므로 `nn.Linear(4, 3)`이면 충분하다.


In [ ]:
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
model = nn.Linear(4, 3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

⑦ **학습 루프**: 배치 단위로 "비우기 $$ 순전파 $$ 손실 $$ 역전파 $$ 갱신"을 반복한다. 한 epoch이 끝날 때마다 검증 정확도를 확인한다.


In [ ]:
for epoch in range(5):
    model.train()
    for xb, yb in train_loader:                 # 배치 단위 반복
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    # ----- 검증 -----
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).argmax(dim=1)
            correct += (pred == yb).sum().item()
            total   += yb.size(0)
    print(f"epoch {epoch+1} | val acc {correct/total:.3f}")

---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
